In [1]:
import os
import re
from datetime import datetime
from pathlib import Path

import ee
import geemap 

import gee_export_helpers as gee_helpers


#####################################
#    SET ALL PARAMETERS #############
#####################################

CYVERSE = False
PROJECT_DIR = "projects/ee-tymc5571-goodfire/assets"
GEE_PROJECT = "ee-tymc5571-goodfire"
DRIVE_FOLDER      = "GEE_Exports_smoke"
CRS = "EPSG:5070"  # export CRS

# ee.Authenticate()
ee.Authenticate(
    auth_mode='notebook',
    scopes=[
        'https://www.googleapis.com/auth/earthengine',
        'https://www.googleapis.com/auth/devstorage.read_write',
        'https://www.googleapis.com/auth/drive'
    ]
)
ee.Initialize(project=GEE_PROJECT)

if CYVERSE:
    SERVICE_FILE = "~/data-store/data/iplant/home/tylermcintosh/secrets/tymc5571-utils-project-692f27c034dd.json"  # Path to service account file
    DIR_DERIVED = "~/data-store/data/iplant/home/shared/earthlab/macrosystems/cbi-module/data/derived/"
else:
    SERVICE_FILE = "C:/Users/tymc5571/dev/cbi-module/config/secrets/tymc5571-utils-project-692f27c034dd.json"  # Path to service account file
    DIR_DERIVED = "C:/Users/tymc5571/dev/cbi-module/data/derived/"

    

Path(DIR_DERIVED).mkdir(parents=True, exist_ok=True)


c:\Users\tymc5571\AppData\Local\miniconda3\envs\cbi-module\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [ ]:
import importlib
importlib.reload(gee_helpers)


{'assets': [{'type': 'IMAGE',
   'name': 'projects/ee-tymc5571-goodfire/assets/FIREDv2_2025_cbi_bc_2000_2004',
   'id': 'projects/ee-tymc5571-goodfire/assets/FIREDv2_2025_cbi_bc_2000_2004',
   'updateTime': '2025-10-09T03:44:47.200546Z'},
  {'type': 'IMAGE',
   'name': 'projects/ee-tymc5571-goodfire/assets/FIREDv2_2025_cbi_bc_2005_2009',
   'id': 'projects/ee-tymc5571-goodfire/assets/FIREDv2_2025_cbi_bc_2005_2009',
   'updateTime': '2025-10-09T04:59:43.326329Z'},
  {'type': 'IMAGE',
   'name': 'projects/ee-tymc5571-goodfire/assets/FIREDv2_2025_cbi_bc_2010_2014',
   'id': 'projects/ee-tymc5571-goodfire/assets/FIREDv2_2025_cbi_bc_2010_2014',
   'updateTime': '2025-10-09T07:14:38.514634Z'},
  {'type': 'IMAGE',
   'name': 'projects/ee-tymc5571-goodfire/assets/FIREDv2_2025_cbi_bc_2015_2019',
   'id': 'projects/ee-tymc5571-goodfire/assets/FIREDv2_2025_cbi_bc_2015_2019',
   'updateTime': '2025-10-10T14:39:08.667249Z'},
  {'type': 'IMAGE',
   'name': 'projects/ee-tymc5571-goodfire/assets/FIRED

In [3]:
# filter to assets of interest

gee_helpers.list_assets(PROJECT_DIR)

assets = gee_helpers.list_assets(PROJECT_DIR)
filtered_assets = [a for a in assets if ("splitfrg5" in a and (m := re.search(r'(\d{4})$', a)) and 2000 <= int(m.group(1)) <= 2010)]
print(filtered_assets)

['projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2000', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2001', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2002', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2003', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2004', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2005', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2006', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2007', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2008', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2009', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2010']


In [27]:
for asset in filtered_assets:
    task = ee.batch.Export.image.toDrive(
        image=ee.Image(asset),
        description=f'gwf_export_{asset[-4:]}',
        folder=DRIVE_FOLDER,
        fileNamePrefix=f'gwf_wumi_{asset[-4:]}',
        #region=west_geom,
        scale=30,
        crs=CRS,
        maxPixels=1e13
    )
    task.start()
    print(f'Started export task for {asset} with task ID: gwf_export_{asset[-4:]}')

Started export task for projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2008 with task ID: gwf_export_2008
Started export task for projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2009 with task ID: gwf_export_2009
Started export task for projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2010 with task ID: gwf_export_2010


In [ ]:
for asset in filtered_assets:

    gee_helpers.download_merge_from_drive(
        description=f'gwf_wumi_{asset[-4:]}',
        local_filename=os.path.join(DIR_DERIVED, f'gwf_wumi_{asset[-4:]}.tif'),
        drive_folder=DRIVE_FOLDER,
        service_account_file=SERVICE_FILE,
        as_cog=False,
        compress="ZSTD",
        predictor=2, # 3 = best for float32 data (like indices or continuous values), 2 = best for integers
        blocksize=512, # tile size; 256–512 is good default
        bigtiff="IF_SAFER",
        n_workers=6,
        check_existing=True,
        verbose=True
    )


Found 121 file(s). Starting download to C:\Users\tymc5571\AppData\Local\Temp\tmpq9wpcr5z ...
Downloaded gwf_wumi_2000-0000000000-0000000000.tif
Downloaded gwf_wumi_2000-0000000000-0000014848.tif
Downloaded gwf_wumi_2000-0000000000-0000022272.tif
Downloaded gwf_wumi_2000-0000000000-0000029696.tif
Downloaded gwf_wumi_2000-0000000000-0000037120.tif
Downloaded gwf_wumi_2000-0000000000-0000044544.tif
Downloaded gwf_wumi_2000-0000000000-0000007424.tif
Downloaded gwf_wumi_2000-0000000000-0000059392.tif
Downloaded gwf_wumi_2000-0000000000-0000074240.tif
Downloaded gwf_wumi_2000-0000000000-0000051968.tif
Downloaded gwf_wumi_2000-0000000000-0000066816.tif
Downloaded gwf_wumi_2000-0000007424-0000000000.tif
Downloaded gwf_wumi_2000-0000007424-0000007424.tif
Downloaded gwf_wumi_2000-0000007424-0000044544.tif
Downloaded gwf_wumi_2000-0000007424-0000022272.tif
Downloaded gwf_wumi_2000-0000007424-0000029696.tif
Downloaded gwf_wumi_2000-0000007424-0000014848.tif
Downloaded gwf_wumi_2000-0000007424-0000